In [ ]:
!pip install --upgrade transformers datasets seqeval accelerate --quiet

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from collections import Counter
from datasets import Dataset
from seqeval.metrics import f1_score, classification_report
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    Trainer,
    TrainingArguments,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
)
import random

In [ ]:
BASE_PATH   = "/content/drive/MyDrive/Frames_experiment"
BIO_FILE    = f"{BASE_PATH}/dataset_bio.conll"
TRAIN_FILE  = f"{BASE_PATH}/train.conll"
DEV_FILE    = f"{BASE_PATH}/dev.conll"
TEST_FILE   = f"{BASE_PATH}/test.conll"

In [ ]:
#READ AND SPLIT BIO DATASET

# Read all sentences
with open(BIO_FILE, encoding="utf8") as f:
    sentences = f.read().strip().split("\n\n")

print("Total sentences:", len(sentences))

# Shuffle sentences
random.shuffle(sentences)

# Split
n = len(sentences)
train_end = int(0.8 * n)
dev_end   = int(0.9 * n)

train_sentences = sentences[:train_end]
dev_sentences   = sentences[train_end:dev_end]
test_sentences  = sentences[dev_end:]

# Write splits
def write_sentences(sent_list, path):
    with open(path, "w", encoding="utf8") as f:
        for s in sent_list:
            f.write(s + "\n\n")

write_sentences(train_sentences, TRAIN_FILE)
write_sentences(dev_sentences, DEV_FILE)
write_sentences(test_sentences, TEST_FILE)

print("Splits created:")
print("Train:", len(train_sentences))
print("Dev:", len(dev_sentences))
print("Test:", len(test_sentences))

In [ ]:
# READ CoNLL FILES INTO PYTHON LISTS

def read_conll(file_path):
    sentences, labels = [], []
    tokens, tags = [], []
    with open(file_path, encoding="utf8") as f:
        for line in f:
            line = line.strip()
            if line == "":
                if tokens:
                    sentences.append(tokens)
                    labels.append(tags)
                    tokens, tags = [], []
            else:
                token, tag = line.split()
                tokens.append(token)
                tags.append(tag)
    if tokens:
        sentences.append(tokens)
        labels.append(tags)
    return sentences, labels

train_tokens, train_tags = read_conll(TRAIN_FILE)
dev_tokens, dev_tags     = read_conll(DEV_FILE)
test_tokens, test_tags   = read_conll(TEST_FILE)

print(f"Train: {len(train_tokens)} | Dev: {len(dev_tokens)} | Test: {len(test_tokens)}")

In [ ]:
#LABEL SETUP

label_list = sorted(set(tag for sent in train_tags for tag in sent))
label2id   = {l: i for i, l in enumerate(label_list)}
id2label   = {i: l for l, i in label2id.items()}
num_labels = len(label_list)

print("Labels:", label_list)

In [ ]:
# SEE LABEL DISTRIBUTION (optional)
from collections import Counter

all_tags = [tag for sent in train_tags for tag in sent if tag != "O"]
counter = Counter(all_tags)

print("Label counts in training data (excluding 'O'):\n")
for label, count in sorted(counter.items()):
    print(f"{label}: {count}")

In [ ]:
#CREATE DATASETS

def make_dataset(tokens, tags):
    return Dataset.from_dict({
        "tokens":   tokens,
        "ner_tags": [[label2id[tag] for tag in sent] for sent in tags]
    })

train_dataset = make_dataset(train_tokens, train_tags)
dev_dataset   = make_dataset(dev_tokens, dev_tags)
test_dataset  = make_dataset(test_tokens, test_tags)

In [ ]:
# TOKENIZER AND ALIGNMENT

MODEL_NAME = "neuralmind/bert-base-portuguese-cased"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

train_dataset = train_dataset.map(tokenize_and_align_labels, batched=True)
dev_dataset   = dev_dataset.map(tokenize_and_align_labels, batched=True)
test_dataset  = test_dataset.map(tokenize_and_align_labels, batched=True)

In [ ]:
# MODEL SETUP

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.2
)

In [ ]:
# CLASS WEIGHTS

tag_counts = Counter(tag for sent in train_tags for tag in sent)
total      = sum(tag_counts.values())

weights = torch.tensor(
    [np.sqrt(total / (tag_counts.get(id2label[i], 1) * num_labels)) for i in range(num_labels)],
    dtype=torch.float
).to("cuda")

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.get("labels")
        outputs = model(**inputs)
        logits  = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=weights, ignore_index=-100)
        loss = loss_fct(logits.view(-1, num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [ ]:
# TRAINING ARGUMENTS

training_args = TrainingArguments(
    output_dir                  = "/content/frame_model",
    learning_rate               = 3e-5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    num_train_epochs            = 15,
    weight_decay                = 0.01,
    warmup_ratio                = 0.15,
    lr_scheduler_type           = "cosine",
    logging_dir                 = "./logs",
    logging_steps               = 50,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1",
    fp16                        = True,
    dataloader_pin_memory       = False,
)

In [ ]:
# METRICS

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    true_predictions = [
        [id2label[pred] for pred, lab in zip(preds, labs) if lab != -100]
        for preds, labs in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[lab] for pred, lab in zip(preds, labs) if lab != -100]
        for preds, labs in zip(predictions, labels)
    ]
    print(classification_report(true_labels, true_predictions))
    return {"f1": f1_score(true_labels, true_predictions)}

In [ ]:
# TRAINER AND TRAINING

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

trainer.train()

In [ ]:
# EVALUATE ON TEST SET

predictions, labels, metrics = trainer.predict(test_dataset)
print("Test metrics:", metrics)

all_tags = [tag for sent in train_tags for tag in sent if tag != "O"]
counter = Counter(all_tags)
for label, count in sorted(counter.items()):
    print(f"{label}: {count}")

In [ ]:
#SAVED MODEL (optional)

# Paths in your repo structure
MODEL_OUTPUT_DIR = "/content/drive/MyDrive/saved_model"

# Save the model
trainer.model.save_pretrained(MODEL_OUTPUT_DIR)

# Save the tokenizer
tokenizer.save_pretrained(MODEL_OUTPUT_DIR)

print("Model and tokenizer saved to:", MODEL_OUTPUT_DIR)